# Training Better Networks: Batch Normalization and Dropout

In the previous notebook, a dense network and a small CNN classified handwritten digits with about 98% accuracy. This notebook adds two tools used in almost every modern network:

* **Dropout**: randomly switching off units during training, which reduces overfitting
* **Batch normalization**: keeping each layer's inputs in a well-behaved range, which makes training faster and more stable

We'll add both to the digit models from the previous notebook: first the dense network, then the CNN, comparing each with and without.

In [ ]:
import copy

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision
from torch.utils.data import TensorDataset, DataLoader, random_split

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print("PyTorch", torch.__version__, "  device:", device)

# Part 1: Digits again, with a validation set

Same MNIST data as before. This time we keep 5,000 training images aside as a **validation set**, so that we can watch for overfitting while training.

We store every image with shape `(1, 28, 28)`, as the CNN needs. The dense models will start with `nn.Flatten()`, so the same data works for both.

In [ ]:
train_set = torchvision.datasets.MNIST(root="data", train=True, download=True)
test_set = torchvision.datasets.MNIST(root="data", train=False, download=True)

train_images = train_set.data.reshape(-1, 1, 28, 28).float() / 255
test_images = test_set.data.reshape(-1, 1, 28, 28).float() / 255

train_full = TensorDataset(train_images, train_set.targets)
train_data, valid_data = random_split(train_full, [55_000, 5_000], generator=torch.Generator().manual_seed(42))

torch.manual_seed(42)
train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=1000)
test_loader = DataLoader(TensorDataset(test_images, test_set.targets), batch_size=1000)
print(len(train_data), "training images,", len(valid_data), "validation,", len(test_images), "test")

The training and evaluation functions are the ones from the previous notebook, with two additions:
* `fit` evaluates on the validation set after every epoch, and records the training and validation loss and accuracy
* it keeps a copy of the parameters from the epoch with the **best validation accuracy** (`model.state_dict()` holds all the parameters, and `copy.deepcopy` makes an independent copy), and restores them at the end

In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        n += len(labels)
    return total_loss / n, correct / n


def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            total_loss += loss_fn(outputs, labels).item() * len(labels)
            correct += (outputs.argmax(dim=1) == labels).sum().item()
            n += len(labels)
    return total_loss / n, correct / n


def fit(model, train_loader, valid_loader, epochs, lr=0.001):
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"train_loss": [], "train_acc": [], "valid_loss": [], "valid_acc": []}
    best_acc, best_state = 0.0, None
    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, loss_fn, optimizer)
        valid_loss, valid_acc = evaluate(model, valid_loader, loss_fn)
        for key, value in zip(history, (train_loss, train_acc, valid_loss, valid_acc)):
            history[key].append(value)
        if valid_acc > best_acc:
            best_acc, best_state = valid_acc, copy.deepcopy(model.state_dict())
        print(f"epoch {epoch + 1:2d}/{epochs}   train loss {train_loss:.4f}  acc {train_acc:.4f}"
              f"   |   valid loss {valid_loss:.4f}  acc {valid_acc:.4f}")
    model.load_state_dict(best_state)
    return history


def plot_histories(histories, title=""):
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
    for i, (name, h) in enumerate(histories.items()):
        epochs = range(1, len(h["train_loss"]) + 1)
        ax[0].plot(epochs, h["train_loss"], "--", color=f"C{i}", label=f"{name} (training)")
        ax[0].plot(epochs, h["valid_loss"], "-", color=f"C{i}", label=f"{name} (validation)")
        ax[1].plot(epochs, h["train_acc"], "--", color=f"C{i}", label=f"{name} (training)")
        ax[1].plot(epochs, h["valid_acc"], "-", color=f"C{i}", label=f"{name} (validation)")
    ax[0].set_ylabel("loss"); ax[1].set_ylabel("accuracy")
    for a in ax:
        a.set_xlabel("epoch"); a.grid(alpha=0.3); a.legend(fontsize=8)
        a.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()


def count_params(model):
    return sum(p.numel() for p in model.parameters())

# Part 2: Dropout and batch normalization

## Dropout

A **dropout** layer, `nn.Dropout(p)`, randomly sets a fraction `p` of its inputs to zero, choosing a different random set for every batch. A unit can't rely on any particular other unit being there, so the network has to spread what it learns across many units. This makes it less likely to memorize the training data, in other words, to **overfit**.

Dropout only happens during **training**. To keep the overall size of the signal the same, the values that survive are multiplied by $1/(1-p)$. During **evaluation**, dropout does nothing.

In [ ]:
dropout = nn.Dropout(p=0.5)
x = torch.ones(10)

dropout.train()
print("training mode:  ", dropout(x))    # about half are zeroed; survivors are scaled by 1/(1 - 0.5) = 2
dropout.eval()
print("evaluation mode:", dropout(x))    # unchanged

## Batch normalization

We standardize the inputs to a network because training works better on values centered near 0 with a similar spread. But after a layer or two, the values reaching the next layer can drift to any scale, and the drift changes as the earlier layers learn.

A **batch normalization** layer standardizes its inputs again, inside the network: for each feature (or, in a CNN, each channel), it subtracts the mean and divides by the standard deviation **of the current batch**. Then it applies a learned scale and shift, so the network can still choose the range it prefers. This usually makes training faster and less sensitive to the learning rate and the initial weights.

* `nn.BatchNorm1d(n)` follows a dense layer with `n` outputs
* `nn.BatchNorm2d(c)` follows a convolutional layer with `c` channels

At **evaluation** time there may be only one image, so there's no batch to compute statistics from. Instead, batch norm uses running averages of the means and standard deviations it saw during training.

In [ ]:
torch.manual_seed(0)
batch_norm = nn.BatchNorm1d(3)
x = torch.randn(64, 3) * torch.tensor([1.0, 10.0, 100.0]) + torch.tensor([5.0, -20.0, 300.0])

print("before: mean", x.mean(dim=0).round(decimals=2), "  std", x.std(dim=0).round(decimals=2))
y = batch_norm(x)   # training mode: uses this batch's statistics
print("after:  mean", y.mean(dim=0).round(decimals=2), "  std", y.std(dim=0).round(decimals=2))

## `model.train()` and `model.eval()` now really matter

Both layers behave differently during training and evaluation. That's exactly what `model.train()` and `model.eval()` switch, and why our `train_one_epoch` and `evaluate` functions call them. Forgetting `model.eval()` before evaluating or predicting is a classic bug: dropout keeps switching off units, and batch norm uses the statistics of whatever batch it's given. We'll see the effect below.

## A dense network, with and without

The plain dense network is the kind we've used so far. The second version adds batch normalization and dropout to each hidden layer, in a common order: `Linear` → `BatchNorm` → `ReLU` → `Dropout`. The `Linear` layers before a batch norm don't need a bias (`bias=False`), because batch norm adds its own shift.

In [ ]:
def plain_dense():
    torch.manual_seed(42)
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(28 * 28, 512), nn.ReLU(),
        nn.Linear(512, 256), nn.ReLU(),
        nn.Linear(256, 10),
    ).to(device)

def dense_bn_dropout():
    torch.manual_seed(42)
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(28 * 28, 512, bias=False), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.2),
        nn.Linear(512, 256, bias=False), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.2),
        nn.Linear(256, 10),
    ).to(device)

dense_histories = {}
print("Plain dense network")
plain_dense_model = plain_dense()
dense_histories["plain"] = fit(plain_dense_model, train_loader, valid_loader, epochs=5)
print("\nWith batch norm + dropout")
dense_bn_model = dense_bn_dropout()
dense_histories["batch norm + dropout"] = fit(dense_bn_model, train_loader, valid_loader, epochs=5)

plot_histories(dense_histories, "Dense networks on MNIST")

Things to look for in the curves (exact numbers vary from run to run and machine to machine):
* How does the **validation** accuracy of the two models compare, especially in the first epochs?
* For the model with dropout, the **training** accuracy may be close to, or even *below*, its validation accuracy. That's not a bug: training accuracy is measured while dropout is switching off a fifth of the units, whereas validation runs in evaluation mode with the full network.
* For the plain model, watch the **gap** between training and validation loss: a widening gap is the sign of overfitting.

### What happens if you forget `model.eval()`?

In [ ]:
loss_fn = nn.CrossEntropyLoss()
_, acc_eval = evaluate(dense_bn_model, test_loader, loss_fn)

dense_bn_model.train()                       # wrongly left in training mode
correct = 0
with torch.no_grad():
    for images, labels in test_loader:
        outputs = dense_bn_model(images.to(device))
        correct += (outputs.argmax(dim=1).cpu() == labels).sum().item()
print(f"test accuracy in evaluation mode: {acc_eval:.4f}")
print(f"test accuracy in training mode:   {correct / len(test_images):.4f}   <- dropout still active, batch statistics used")

The accuracy in training mode is usually lower, and it's random: run the cell again and you'll get a slightly different number, because dropout switches off different units each time.

## A CNN, with and without

The plain CNN is the one from the previous notebook. The second version has **two** convolutional blocks, each with batch normalization (`nn.BatchNorm2d`), and dropout before the dense layers.

In [ ]:
def plain_cnn():
    torch.manual_seed(42)
    return nn.Sequential(
        nn.Conv2d(1, 32, kernel_size=3, padding=1), nn.ReLU(),
        nn.MaxPool2d(2),                                        # 28 -> 14
        nn.Flatten(),
        nn.Linear(32 * 14 * 14, 64), nn.ReLU(),
        nn.Linear(64, 10),
    ).to(device)

def cnn_bn_dropout():
    torch.manual_seed(42)
    return nn.Sequential(
        nn.Conv2d(1, 32, kernel_size=3, padding=1, bias=False), nn.BatchNorm2d(32), nn.ReLU(),
        nn.MaxPool2d(2),                                        # 28 -> 14
        nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False), nn.BatchNorm2d(64), nn.ReLU(),
        nn.MaxPool2d(2),                                        # 14 -> 7
        nn.Flatten(),
        nn.Dropout(0.25),
        nn.Linear(64 * 7 * 7, 128), nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(128, 10),
    ).to(device)

cnn_histories = {}
print("Plain CNN")
plain_cnn_model = plain_cnn()
cnn_histories["plain"] = fit(plain_cnn_model, train_loader, valid_loader, epochs=5)
print("\nCNN with batch norm + dropout")
cnn_bn_model = cnn_bn_dropout()
cnn_histories["batch norm + dropout"] = fit(cnn_bn_model, train_loader, valid_loader, epochs=5)

plot_histories(cnn_histories, "CNNs on MNIST")

In [ ]:
print(f"{'model':38s} {'parameters':>11s} {'test accuracy':>14s}")
for name, model in [("dense", plain_dense_model), ("dense + batch norm + dropout", dense_bn_model),
                    ("CNN", plain_cnn_model), ("CNN (2 blocks) + batch norm + dropout", cnn_bn_model)]:
    _, acc = evaluate(model, test_loader, loss_fn)
    print(f"{name:38s} {count_params(model):11,d} {acc:14.4f}")

On MNIST every model is already near the top, so the differences are small, a few tenths of a percent. (The deeper CNN also gains from its second convolutional block, not only from batch norm and dropout.) These tools matter much more on harder problems with less data, where overfitting is a real risk.

# Summary

* **Dropout** randomly switches off units during training, so the network can't rely on any one unit. It reduces overfitting.
* **Batch normalization** re-standardizes values inside the network using each batch's statistics, which makes training faster and more stable. Use `nn.BatchNorm1d` after dense layers and `nn.BatchNorm2d` after convolutional layers.
* A common order in each hidden layer: `Linear` (or `Conv2d`) → `BatchNorm` → `ReLU` → `Dropout`. A layer followed by batch norm doesn't need a bias.
* Both behave differently in training and evaluation, so **`model.train()` and `model.eval()` matter**: forgetting `model.eval()` gives worse, random predictions.
* Watch the **gap** between training and validation accuracy and loss: a widening gap means overfitting.